# LoRA Motion Style — Results Review
Experiment console: metrics, GIF gallery, A/B comparison, loss curves.

In [ ]:
import json
import os
from pathlib import Path
from IPython.display import Image, display, HTML
import ipywidgets as widgets
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Paths on training machine (for loss curves / remote eval)
EVAL_BASE  = Path("/transfer/loraoutputs/eval")
MODEL_BASE = Path("/transfer/loraoutputs/models")

# Local output folder (GIFs + evaluation_results.json copied from training machine)
LOCAL_BASE = Path("outputs")

## 1. Experiment metadata

In [ ]:
VERSIONS = {
    "v1": {"alpha": 16, "foot_vel": 0.0, "root_stable": 0.0, "lr": 2e-4, "styles": 5,
           "note": "baseline"},
    "v2": {"alpha": 16, "foot_vel": 2.0, "root_stable": 0.0, "lr": 2e-4, "styles": 5,
           "note": "+ foot velocity penalty"},
    "v3": {"alpha": 8,  "foot_vel": 2.0, "root_stable": 0.0, "lr": 2e-4, "styles": 5,
           "note": "+ lower alpha (scaling 0.5)"},
    "v4": {"alpha": 8,  "foot_vel": 2.0, "root_stable": 1.0, "lr": 1e-4, "styles": 5,
           "note": "+ root stability, lower lr, better contact labels"},
    "v5": {"alpha": 8,  "foot_vel": 2.0, "root_stable": 1.0, "lr": 1e-4, "styles": 20,
           "note": "20 styles (old selection)"},
    "v6": {"alpha": 8,  "foot_vel": 2.0, "root_stable": 1.0, "lr": 1e-4, "styles": 21,
           "note": "21 styles (updated selection: replaced cat/dinosaur/etc.)"},
}

STYLES_V6 = [
    "zombie", "elated", "old", "depressed", "drunk",
    "angry", "chicken", "proud",
    "heavyset", "bigsteps", "stiff", "duckfoot",
    "highknees", "flapping", "punch", "wildarms",
    "handsinpockets", "handsbetweenlegs", "onphoneleft",
    "penguin", "robot",
]

pd.DataFrame(VERSIONS).T.rename_axis("version")

## 2. Quantitative results

In [ ]:
def load_eval(version: str) -> pd.DataFrame:
    """Load evaluation_results.json from local outputs folder."""
    suffix = "" if version == "v1" else f"_{version}"
    path = LOCAL_BASE / f"multi_style{suffix}" / "evaluation_results.json"
    if not path.exists():
        # fallback to remote path
        path = EVAL_BASE / f"multi_style{suffix}" / "evaluation_results.json"
    if not path.exists():
        return pd.DataFrame()
    with open(path) as f:
        data = json.load(f)
    rows = []
    for key, metrics in data.items():
        style = key.replace("lora_", "") if key != "base_model" else "base"
        rows.append({"style": style, **metrics})
    return pd.DataFrame(rows).set_index("style")


eval_data = {v: load_eval(v) for v in VERSIONS}

# Show v6 (latest)
df_latest = eval_data.get("v6", pd.DataFrame())
if not df_latest.empty:
    display(
        df_latest.sort_values("jitter_mean").style
        .background_gradient(subset=["diversity"], cmap="Blues")
        .background_gradient(subset=["jitter_mean"], cmap="Reds_r")
        .format("{:.2f}")
    )

In [ ]:
# Cross-version comparison for a single style
COMPARE_STYLE = "zombie"

rows = []
for v, df in eval_data.items():
    if df.empty or COMPARE_STYLE not in df.index:
        continue
    rows.append({"version": v, **df.loc[COMPARE_STYLE].to_dict()})

if rows:
    print(f"Cross-version: {COMPARE_STYLE}")
    display(pd.DataFrame(rows).set_index("version").style.format("{:.3f}"))

## 3. GIF gallery

In [ ]:
PROMPT_LABELS = [
    "a_person_walking_forward",
    "a_person_walking_in_a_circle",
    "a_person_stepping_sideways",
    "a_person_turning_around",
    "a_person_walking_and_then_stop",
]

def gif_path(version: str, style: str, prompt_idx: int = 0, kind: str = "cmp") -> Path:
    label = PROMPT_LABELS[prompt_idx]
    if version == "v1":
        base_dir = LOCAL_BASE / "generated"
    elif version in ("v2", "v3"):
        # v2/v3 were stored directly in generated/
        base_dir = LOCAL_BASE / "generated"
    else:
        base_dir = LOCAL_BASE / "generated" / version

    # Also check multi_style_vX/viz/ layout
    suffix = "" if version == "v1" else f"_{version}"
    alt_dir = LOCAL_BASE / f"multi_style{suffix}" / "viz"

    if kind == "cmp":
        name = f"{prompt_idx}_cmp_{style}_{label}.gif"
    elif kind == "base":
        name = f"{prompt_idx}_base_{label}.gif"
    else:
        name = f"{prompt_idx}_lora_{style}_{label}.gif"

    for d in [alt_dir, base_dir]:
        p = d / name
        if p.exists():
            return p
    return alt_dir / name  # return expected path even if missing


def show_gif(path: Path, width: int = 380, label: str = "") -> str:
    if path.exists():
        return (f'<div style="display:inline-block;text-align:center;margin:4px">'
                f'<p style="font-size:11px;margin:2px">{label}</p>'
                f'<img src="{path}" width="{width}"></div>')
    return (f'<div style="width:{width}px;display:inline-block;'
            f'color:#bbb;font-size:10px;border:1px dashed #ddd;padding:4px">'
            f'missing<br>{path.name}</div>')


print("GIF helpers loaded.")

In [ ]:
# One style across versions
GALLERY_STYLE    = "zombie"
GALLERY_PROMPT   = 0
GALLERY_VERSIONS = ["v1", "v3", "v5", "v6"]

html = f"<h3>{GALLERY_STYLE} — prompt {GALLERY_PROMPT} — across versions</h3>"
html += '<div style="display:flex;flex-wrap:wrap">'
for v in GALLERY_VERSIONS:
    html += show_gif(gif_path(v, GALLERY_STYLE, GALLERY_PROMPT, "cmp"),
                     width=400, label=f"{v}: {VERSIONS[v]['note']}")
html += '</div>'
display(HTML(html))

In [ ]:
# One version, all styles
MULTI_VER    = "v6"
MULTI_PROMPT = 0
STYLES_SHOW  = ["zombie", "drunk", "old", "chicken", "bigsteps", "robot",
                "flapping", "angry", "proud", "wildarms", "punch", "penguin"]

html = f"<h3>{MULTI_VER} — all styles — prompt {MULTI_PROMPT}</h3>"
html += '<div style="display:flex;flex-wrap:wrap">'
for s in STYLES_SHOW:
    html += show_gif(gif_path(MULTI_VER, s, MULTI_PROMPT, "cmp"), width=320, label=s)
html += '</div>'
display(HTML(html))

## 4. A/B interactive comparison

In [ ]:
style_dd  = widgets.Dropdown(options=STYLES_V6, value="zombie",   description="Style:")
ver_dd    = widgets.Dropdown(options=list(VERSIONS.keys()), value="v6", description="Version:")
prompt_dd = widgets.Dropdown(
    options=[("walking forward", 0), ("walking in circle", 1),
             ("stepping sideways", 2), ("turning around", 3), ("walk then stop", 4)],
    value=0, description="Prompt:"
)
out = widgets.Output()

def update(_=None):
    out.clear_output(wait=True)
    s, v, p = style_dd.value, ver_dd.value, prompt_dd.value
    html = f"<h3>{s} | {v} | prompt {p}: {PROMPT_LABELS[p].replace('_', ' ')}</h3>"
    html += '<div style="display:flex;flex-wrap:wrap">'
    html += show_gif(gif_path(v, s, p, "base"), 380, "Base model")
    html += show_gif(gif_path(v, s, p, "lora"), 380, f"LoRA {v}")
    html += show_gif(gif_path(v, s, p, "cmp"),  760, "Side-by-side")
    html += '</div>'
    with out:
        display(HTML(html))

for w in [style_dd, ver_dd, prompt_dd]:
    w.observe(update, names="value")

update()
display(widgets.HBox([style_dd, ver_dd, prompt_dd]), out)

## 5. Loss curves

In [ ]:
def load_loss_csv(version: str, style: str) -> pd.DataFrame | None:
    suffix = "" if version == "v1" else f"_{version}"
    path = MODEL_BASE / f"lora_bvh_{style}{suffix}" / "loss.csv"
    return pd.read_csv(path) if path.exists() else None


LOSS_VERSION = "v6"
LOSS_STYLES  = ["zombie", "drunk", "old", "chicken", "robot", "bigsteps"]

fig, axes = plt.subplots(1, len(LOSS_STYLES),
                         figsize=(3.5 * len(LOSS_STYLES), 3), sharey=False)
for ax, style in zip(axes, LOSS_STYLES):
    df_loss = load_loss_csv(LOSS_VERSION, style)
    ax.set_title(style, fontsize=10)
    if df_loss is not None:
        ax.plot(df_loss["step"], df_loss["loss"], linewidth=1)
        ax.set_xlabel("step", fontsize=8)
        ax.set_ylabel("loss", fontsize=8)
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
    else:
        ax.text(0.5, 0.5, "no loss.csv", ha="center", va="center",
                transform=ax.transAxes, color="#aaa", fontsize=9)

fig.suptitle(f"Training loss — {LOSS_VERSION}", fontsize=11)
plt.tight_layout()
plt.show()

## 6. Cross-version metrics bar chart

In [ ]:
CHART_STYLES  = ["zombie", "drunk", "old", "robot", "chicken", "bigsteps"]
CHART_METRIC  = "jitter_mean"   # or 'diversity'
CHART_VERSIONS = ["v1", "v3", "v5", "v6"]

rows = []
for v in CHART_VERSIONS:
    df = eval_data.get(v, pd.DataFrame())
    if df.empty:
        continue
    for s in CHART_STYLES:
        if s in df.index and CHART_METRIC in df.columns:
            rows.append({"version": v, "style": s, CHART_METRIC: df.loc[s, CHART_METRIC]})

if rows:
    pivot = pd.DataFrame(rows).pivot(index="style", columns="version", values=CHART_METRIC)
    ax = pivot.plot(kind="bar", figsize=(10, 4), width=0.7)
    ax.set_title(f"{CHART_METRIC} by style and version")
    ax.set_ylabel(CHART_METRIC)
    ax.set_xlabel("")
    ax.legend(title="version", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No eval data loaded. Check that evaluation_results.json files are in outputs/multi_style_vX/")